In [2]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [3]:
df_05 = pd.DataFrame(pd.read_csv("D:/My_projects/Capstone_project/data/raw/05_category_inflows.csv"))
df_05

,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0
...,...,...,...
139,2025-03,Sectoral/Thematic,8614.0
140,2025-03,Liquid,38681.0
141,2025-03,Short Duration,4886.0
142,2025-03,Gilt,956.0


In [4]:
df_05.shape

(144, 3)

In [5]:
df_05.columns

Index(['month', 'category', 'net_inflow_crore'], dtype='str')

In [6]:
df_05.dtypes

month                   str
category                str
net_inflow_crore    float64
dtype: object

In [7]:
df_05.head()

,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0


In [8]:
df_05.tail()

,month,category,net_inflow_crore
139,2025-03,Sectoral/Thematic,8614.0
140,2025-03,Liquid,38681.0
141,2025-03,Short Duration,4886.0
142,2025-03,Gilt,956.0
143,2025-03,Hybrid,2830.0


In [9]:
df_05.info()

<class 'pandas.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   month             144 non-null    str    
 1   category          144 non-null    str    
 2   net_inflow_crore  144 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.5 KB


In [10]:
df_05.describe(include="all")

,month,category,net_inflow_crore
count,144,144,144.000000
unique,12,12,NaN
top,2024-04,Large Cap,NaN
freq,12,12,NaN
mean,NaN,NaN,6473.881944
std,NaN,NaN,9718.335947
min,NaN,NaN,437.000000
25%,NaN,NaN,1801.250000
50%,NaN,NaN,4148.000000
75%,NaN,NaN,5304.000000


In [11]:
df_05.isna().sum()

month               0
category            0
net_inflow_crore    0
dtype: int64

In [12]:
df_05.duplicated().sum()

np.int64(0)

In [15]:
df_05['month'] = pd.to_datetime(
    df_05['month'], format='%Y-%m'
)

print("Start:", df_05['month'].min())
print("End:", df_05['month'].max())

print("\nCategories:")
print(df_05['category'].unique())

print("\nTotal categories:", df_05['category'].nunique())

Start: 2024-04-01 00:00:00
End: 2025-03-01 00:00:00

Categories:
<StringArray>
[        'Large Cap',           'Mid Cap',         'Small Cap',
         'Flexi Cap',   'Large & Mid Cap',              'ELSS',
      'Value/Contra', 'Sectoral/Thematic',            'Liquid',
    'Short Duration',              'Gilt',            'Hybrid']
Length: 12, dtype: str

Total categories: 12


In [16]:
category_summary = df_05.groupby('category').agg(
    total_net_inflow=('net_inflow_crore', 'sum'),
    avg_monthly_inflow=('net_inflow_crore', 'mean')
).reset_index()

category_summary = category_summary.sort_values(
    by='total_net_inflow',
    ascending=False
)

category_summary

,category,total_net_inflow,avg_monthly_inflow
6,Liquid,451275.0,37606.250000
8,Sectoral/Thematic,103829.0,8652.416667
1,Flexi Cap,63989.0,5332.416667
4,Large & Mid Cap,57752.0,4812.666667
9,Short Duration,55530.0,4627.500000
7,Mid Cap,55312.0,4609.333333
10,Small Cap,46596.0,3883.000000
3,Hybrid,38868.0,3239.000000
5,Large Cap,25633.0,2136.083333
11,Value/Contra,16980.0,1415.000000


In [17]:
fig = px.bar(
    category_summary,
    x='total_net_inflow',
    y='category',
    orientation='h',
    text='total_net_inflow',
    title='Total Net Inflows by Mutual Fund Category (Apr 2024–Mar 2025)',
    labels={
        'total_net_inflow': 'Total Net Inflow (₹ Crore)',
        'category': 'Fund Category'
    }
)

fig.update_traces(
    texttemplate='₹%{x:,.0f} Cr',
    textposition='outside'
)

fig.update_layout(
    template='plotly_white',
    width=1200,
    height=700,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [18]:
fig.write_image("../reports/plots/category_wise_net_inflows.png")

In [19]:
fig = px.line(
    df_05,
    x='month',
    y='net_inflow_crore',
    color='category',
    markers=True,
    title='Monthly Net Inflow Trends by Mutual Fund Category',
    labels={
        'month': 'Month',
        'net_inflow_crore': 'Net Inflow (₹ Crore)',
        'category': 'Fund Category'
    }
)

fig.update_layout(
    template='plotly_white',
    width=1300,
    height=750,
    hovermode='x unified'
)

fig.show()

In [20]:
fig.write_image("../reports/plots/monthly_category_inflow_trends.png")

In [21]:
category_month_extremes = df_05.loc[
    df_05.groupby('category')['net_inflow_crore'].idxmax(),
    ['category', 'month', 'net_inflow_crore']
].rename(columns={
    'month': 'highest_inflow_month',
    'net_inflow_crore': 'highest_inflow'
})

lowest = df_05.loc[
    df_05.groupby('category')['net_inflow_crore'].idxmin(),
    ['category', 'month', 'net_inflow_crore']
].rename(columns={
    'month': 'lowest_inflow_month',
    'net_inflow_crore': 'lowest_inflow'
})

category_month_extremes = category_month_extremes.merge(
    lowest, on='category'
)

category_month_extremes

,category,highest_inflow_month,highest_inflow,lowest_inflow_month,lowest_inflow
0,ELSS,2024-11-01,571.0,2025-02-01,437.0
1,Flexi Cap,2024-11-01,6111.0,2024-06-01,4478.0
2,Gilt,2024-07-01,959.0,2024-11-01,704.0
3,Hybrid,2024-08-01,3684.0,2025-03-01,2830.0
4,Large & Mid Cap,2024-11-01,5556.0,2024-04-01,4214.0
5,Large Cap,2024-07-01,2574.0,2024-11-01,1870.0
6,Liquid,2024-08-01,41952.0,2025-02-01,32374.0
7,Mid Cap,2024-05-01,5300.0,2024-04-01,3897.0
8,Sectoral/Thematic,2024-06-01,10030.0,2024-11-01,7397.0
9,Short Duration,2024-09-01,5327.0,2025-02-01,4033.0


In [22]:
monthly_total = df_05.groupby('month').agg(
    total_monthly_inflow=('net_inflow_crore', 'sum')
).reset_index()

monthly_total = monthly_total.sort_values(
    'total_monthly_inflow',
    ascending=False
)

monthly_total

,month,total_monthly_inflow
1,2024-05-01,82661.0
4,2024-08-01,81601.0
2,2024-06-01,80911.0
7,2024-11-01,80323.0
6,2024-10-01,79180.0
11,2025-03-01,78573.0
8,2024-12-01,75943.0
5,2024-09-01,75865.0
3,2024-07-01,75608.0
0,2024-04-01,74526.0


In [23]:
monthly_total = monthly_total.sort_values('month')

fig = px.line(
    monthly_total,
    x='month',
    y='total_monthly_inflow',
    markers=True,
    title='Monthly Total Net Inflows Across Mutual Fund Categories',
    labels={
        'month': 'Month',
        'total_monthly_inflow': 'Total Net Inflow (₹ Crore)'
    }
)

fig.update_layout(
    template='plotly_white',
    width=1200,
    height=600,
    hovermode='x unified'
)

fig.show()

In [24]:
fig.write_image("../reports/plots/monthly_total_category_inflows.png")

In [25]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

df_05.to_csv(
    processed_path / "category_inflowsr.csv",
    index=False
)